# 13 — Cluster Descriptions, Category Mapping, and Semantic Evaluation

Three sections:
1. **LexRank descriptions** — for every clustering method × cluster, summarise
   all articles in the cluster to 2-3 representative sentences using LexRank.
2. **Semantic category mapping** — embed each cluster description with
   `all-mpnet-base-v2`, find the most similar of the 16 canonical category
   descriptions by cosine similarity.
3. **Semantic evaluation** — report a per-method semantic similarity score
   (mean cosine similarity across all 16 clusters) alongside the existing
   Hungarian-ACC, producing a combined comparison table.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from utils import config
from utils.cluster_description import generate_cluster_descriptions

train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
summaries   = train_clean["summary"].tolist()

# Discover all clustering methods that have saved label arrays
label_files = sorted(config.RESULTS_DIR.glob("cluster_labels_*.npy"))
ALL_METHODS = [f.stem.replace("cluster_labels_", "") for f in label_files]
print(f"Found {len(ALL_METHODS)} clustering methods:")
for m in ALL_METHODS:
    print(f"  {m}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 1 — LexRank cluster descriptions
# ════════════════════════════════════════════════════════════════════
all_descriptions = {}   # method -> {cluster_id(int): description(str)}

for method in ALL_METHODS:
    out_path = config.RESULTS_DIR / f"cluster_descriptions_{method}.json"

    if out_path.exists():
        # Load from disk — don't re-run LexRank (slow)
        with open(out_path, encoding="utf-8") as f:
            raw = json.load(f)
        all_descriptions[method] = {int(k): v for k, v in raw.items()}
        print(f"  {method}: loaded from cache ({len(all_descriptions[method])} clusters)")
        continue

    labels_path = config.RESULTS_DIR / f"cluster_labels_{method}.npy"
    cluster_labels = np.load(labels_path)

    descs = generate_cluster_descriptions(summaries, cluster_labels, n_sentences=3, min_words=10)
    all_descriptions[method] = descs

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in descs.items()}, f, indent=2, ensure_ascii=False)

    print(f"  {method}: {len(descs)} cluster descriptions saved → {out_path.name}")

print(f"\nSection 1 complete. {len(ALL_METHODS)} methods processed.")

In [ ]:
# Print a sample description for cluster 0 of each method
print("=== Sample descriptions (cluster 0 per method) ===\n")
for method in ALL_METHODS[:5]:   # show first 5 to avoid scrolling
    d = all_descriptions[method].get(0, "(missing)")
    print(f"[{method}]\n{d}\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 2 — Semantic category mapping
# ════════════════════════════════════════════════════════════════════

# 16 canonical one-sentence descriptions, one per class in config.CLASS_NAMES order
CANONICAL_DESCRIPTIONS = {
    "ARTS & CULTURE":  "Articles about visual arts, performing arts, music, literature, cultural events, and creative expression.",
    "BUSINESS":        "Articles about corporate finance, markets, economics, trade, and business strategy.",
    "COMEDY":          "Articles about stand-up comedy, satirical shows, late-night television hosts, and humorous commentary on current events.",
    "CRIME":           "Articles about criminal investigations, arrests, court cases, and law enforcement activities.",
    "EDUCATION":       "Articles about schools, universities, student loans, teachers, and educational policy.",
    "ENTERTAINMENT":   "Articles about movies, television shows, celebrity news, awards ceremonies, and pop culture.",
    "ENVIRONMENT":     "Articles about climate change, environmental policy, wildlife, natural disasters, and conservation.",
    "HEALTH":          "Articles about medical research, public health, disease outbreaks, vaccines, and healthcare policy.",
    "MEDIA":           "Articles about journalism, news organisations, television networks, media industry controversies, and press freedom.",
    "NEWS":            "Breaking news and general current events not classified under a more specific category.",
    "POLITICS":        "Articles about elections, government policy, politicians, legislation, and political parties.",
    "RELIGION":        "Articles about religious institutions, faith, spirituality, religious leaders, and interfaith issues.",
    "SCIENCE":         "Articles about scientific discoveries, space exploration, NASA, astronomy, physics, and technology research.",
    "SPORTS":          "Articles about professional and amateur sports competitions, athletes, teams, tournaments, and sporting events.",
    "TECH":            "Articles about consumer technology, gadgets, software, social media platforms, and the tech industry.",
    "WOMEN":           "Articles about women's rights, gender equality, sexual harassment, feminism, and women in public life.",
}

assert list(CANONICAL_DESCRIPTIONS.keys()) == config.CLASS_NAMES, (
    "CANONICAL_DESCRIPTIONS keys must match config.CLASS_NAMES order"
)

print("Loading all-mpnet-base-v2 for semantic embeddings...")
embed_model = SentenceTransformer("all-mpnet-base-v2")
category_texts = list(CANONICAL_DESCRIPTIONS.values())
category_names = list(CANONICAL_DESCRIPTIONS.keys())

category_embs = embed_model.encode(
    category_texts, batch_size=16, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=True,
).astype(np.float32)

print(f"Category embeddings: {category_embs.shape}")

In [ ]:
all_mappings = {}   # method -> {cluster_id(int): {"mapped_category": str, "similarity": float}}

for method in ALL_METHODS:
    descs = all_descriptions[method]   # {int: str}
    cluster_ids = sorted(descs.keys())
    desc_texts  = [descs[c] for c in cluster_ids]

    cluster_embs = embed_model.encode(
        desc_texts, batch_size=16, show_progress_bar=False,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)

    # Both arrays are L2-normalised → dot product == cosine similarity
    sim_matrix = cluster_embs @ category_embs.T   # (n_clusters, 16)

    mapping = {}
    for i, c in enumerate(cluster_ids):
        best_cat_idx = int(np.argmax(sim_matrix[i]))
        mapping[c] = {
            "mapped_category": category_names[best_cat_idx],
            "similarity": float(sim_matrix[i, best_cat_idx]),
        }
    all_mappings[method] = mapping

    out_path = config.RESULTS_DIR / f"cluster_category_mapping_{method}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in mapping.items()}, f, indent=2, ensure_ascii=False)

    print(f"  {method}: mappings saved → {out_path.name}")

print(f"\nSection 2 complete. {len(ALL_METHODS)} methods processed.")

In [ ]:
# Spot-check: print cluster → category table for best method
best_method = "mpnet_kmeans"
mapping = all_mappings[best_method]
descs   = all_descriptions[best_method]

clusters_csv = pd.read_csv(config.RESULTS_DIR / f"clusters_{best_method}.csv")
majority_map = dict(zip(clusters_csv["cluster"], clusters_csv["majority_true_label"]))

print(f"{'Cluster':>8} {'Majority True':>20} {'Mapped Category':>20} {'Sim':>6} {'Match':>6}")
print("-" * 65)
for c in sorted(mapping.keys()):
    m = mapping[c]
    majority = majority_map.get(c, "?")
    match = "✓" if m["mapped_category"] == majority else "✗"
    print(f"{c:>8} {majority:>20} {m['mapped_category']:>20} {m['similarity']:>6.3f} {match:>6}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 3 — Semantic evaluation
# ════════════════════════════════════════════════════════════════════
import json as _json

rows = []
for method in ALL_METHODS:
    metrics_path = config.RESULTS_DIR / f"metrics_{method}.json"
    if not metrics_path.exists():
        print(f"  WARNING: no metrics file for {method}, skipping")
        continue
    metrics = _json.load(open(metrics_path))

    clusters_path = config.RESULTS_DIR / f"clusters_{method}.csv"
    clusters_df   = pd.read_csv(clusters_path)
    majority_map  = dict(zip(clusters_df["cluster"], clusters_df["majority_true_label"]))

    mapping = all_mappings.get(method, {})

    similarities = [v["similarity"] for v in mapping.values()]
    mean_sem = float(np.mean(similarities)) if similarities else 0.0

    n_match = sum(
        1 for c, v in mapping.items()
        if v["mapped_category"] == majority_map.get(c, "")
    )
    map_acc = n_match / len(mapping) if mapping else 0.0

    rows.append({
        "method":              method,
        "ACC_hungarian":       round(metrics["ACC (Hungarian)"], 4),
        "macro_f1":            round(metrics["Macro F1"], 4),
        "NMI":                 round(metrics["NMI"], 4),
        "mean_semantic_score": round(mean_sem, 4),
        "mapping_accuracy":    round(map_acc, 4),
    })

eval_df = pd.DataFrame(rows).sort_values("mean_semantic_score", ascending=False)
out_csv = config.RESULTS_DIR / "semantic_evaluation.csv"
eval_df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")

In [ ]:
print("\n=== SEMANTIC EVALUATION — All Methods ===\n")
print(f"{'Method':<35} {'ACC':>7} {'F1':>7} {'NMI':>7} {'SemanticSim':>12} {'MapACC':>8}")
print("-" * 82)
for _, row in eval_df.iterrows():
    print(
        f"{row['method']:<35} {row['ACC_hungarian']:>7.4f} "
        f"{row['macro_f1']:>7.4f} {row['NMI']:>7.4f} "
        f"{row['mean_semantic_score']:>12.4f} {row['mapping_accuracy']:>8.4f}"
    )

assert len(eval_df) >= len(ALL_METHODS) - 1, (
    "Fewer rows than expected — check for missing metrics files"
)

check = pd.read_csv(config.RESULTS_DIR / "semantic_evaluation.csv")
assert list(check.columns) == [
    "method", "ACC_hungarian", "macro_f1", "NMI",
    "mean_semantic_score", "mapping_accuracy"
], f"Unexpected columns: {list(check.columns)}"
assert len(check) >= 14, f"Expected at least 14 rows, got {len(check)}"

top_method = eval_df.iloc[0]["method"]
print(f"\nHighest semantic similarity: {top_method} ({eval_df.iloc[0]['mean_semantic_score']:.4f})")
print(f"Highest ACC: {eval_df.sort_values('ACC_hungarian', ascending=False).iloc[0]['method']}")
print(f"\nsemantic_evaluation.csv: {len(check)} rows, columns OK")